# Pre-Processing

In [13]:
import numpy as np
import scipy.sparse as sp
from collections import defaultdict
import os


In [ ]:
DATA_PATH = "/Users/hemanthsai/Downloads/train-1.txt"
print("Using dataset:", DATA_PATH)
print("File exists:", os.path.exists(DATA_PATH))


Using dataset: /Users/hemanthsai/Downloads/train-1.txt
File exists: True


In [15]:
def read_interactions(path):
    user2idx = {}
    item2idx = {}

    idx2user = []
    idx2item = []

    rows = []
    cols = []
    data = []

    user_items = defaultdict(set)

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue

            user_id = parts[0]
            item_ids = parts[1:]

            # Encode user
            if user_id not in user2idx:
                user2idx[user_id] = len(idx2user)
                idx2user.append(user_id)
            u = user2idx[user_id]

            # Encode items
            for item_id in item_ids:
                if item_id not in item2idx:
                    item2idx[item_id] = len(idx2item)
                    idx2item.append(item_id)
                i = item2idx[item_id]

                rows.append(u)
                cols.append(i)
                data.append(1)
                user_items[u].add(i)

    num_users = len(idx2user)
    num_items = len(idx2item)

    interaction_matrix = sp.coo_matrix(
        (data, (rows, cols)),
        shape=(num_users, num_items)
    ).tocsr()

    return interaction_matrix, idx2user, idx2item, user_items, user2idx, item2idx


In [16]:
interaction_matrix, idx2user, idx2item, user_items, user2idx, item2idx = read_interactions(DATA_PATH)

print("Preprocessing complete!")
print("------------------------------------")
print("Users:", len(idx2user))
print("Items:", len(idx2item))
print("Interactions:", interaction_matrix.nnz)
print("Matrix shape:", interaction_matrix.shape)


Preprocessing complete!
------------------------------------
Users: 31668
Items: 38048
Interactions: 1237259
Matrix shape: (31668, 38048)


In [17]:
def split_train_val(user_items, val_ratio=0.1, seed=42):
    rng = np.random.default_rng(seed)
    train = defaultdict(set)
    val = defaultdict(set)

    for u, items in user_items.items():
        items_list = list(items)

        if len(items_list) <= 1:
            train[u] = set(items_list)
            continue

        n_val = max(1, int(len(items_list) * val_ratio))
        val_idx = rng.choice(len(items_list), size=n_val, replace=False)

        val_items = {items_list[i] for i in val_idx}
        train_items = set(items_list) - val_items

        if len(train_items) == 0:
            train_items = {items_list[0]}
            val_items = set(items_list[1:])

        train[u] = train_items
        val[u] = val_items

    return train, val


In [18]:
train_user_items, val_user_items = split_train_val(user_items, val_ratio=0.1)

print("Train/Validation split complete!")
print("------------------------------------")
print("Example user 0 train items:", list(train_user_items[0])[:10])
print("Example user 0 val items:", list(val_user_items[0])[:10])


Train/Validation split complete!
------------------------------------
Example user 0 train items: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Example user 0 val items: [1]
